In [1]:
#plot values
import numpy as np
import matplotlib.pyplot as plt
import os

def plot_values(data_path):
    # Get the directory of the current script
    script_dir = os.path.dirname(os.path.abspath(__file__))
    # Construct the path relative to the script location
    data_path = os.path.join(script_dir, data_path)

    # Check if file exists
    if not os.path.exists(data_path):
        print(f"Error: File not found at {data_path}")
        exit()

    try:
        data = np.genfromtxt(data_path, delimiter=" ", skip_header=1)
    except Exception as e:
        print(f"Error loading data: {e}")
        exit()

    print(data)  # Print the loaded data to verify its structure

    # Verify data has correct shape
    if data.ndim != 2 or data.shape[1] < 3:
        print(f"Error: Data shape is incorrect. Expected (n, 3), got {data.shape}")
        exit()

    time = np.arange(len(data)) / 30.  # Assuming each row corresponds to a frame at 30 fps
    red_channel = data[:, 0]
    green_channel = data[:, 1]
    blue_channel = data[:, 2]   

    plt.figure(figsize=(10, 6))
    plt.plot(time, red_channel, label='Red Channel', color='red')
    plt.plot(time, green_channel, label='Green Channel', color='green')
    plt.plot(time, blue_channel, label='Blue Channel', color='blue')   
    plt.title('Color Channel Intensities Over Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Mean Intensity')
    plt.legend()
    plt.grid()
    plt.show()

In [3]:
#extract colors
import subprocess
import numpy as np
import os


def _trim_output_file(output_path, fps, remove_first_n_seconds=0, remove_last_n_seconds=0):
    data = np.loadtxt(output_path)
    if data.ndim == 1:
        data = data[np.newaxis, :]

    first_frames = int(max(0, remove_first_n_seconds) * fps)
    last_frames = int(max(0, remove_last_n_seconds) * fps)

    start_idx = first_frames
    end_idx = data.shape[0] - last_frames if last_frames > 0 else data.shape[0]
    end_idx = max(start_idx, end_idx)

    trimmed = data[start_idx:end_idx, :]
    np.savetxt(output_path, trimmed)

    if first_frames > 0:
        print(f"Removed first {remove_first_n_seconds} seconds ({first_frames} frames)")
    if last_frames > 0:
        print(f"Removed last {remove_last_n_seconds} seconds ({last_frames} frames)")

def run_video_roi_extraction(video_path, output_path, remove_first_n_seconds=0, remove_last_n_seconds=0, fps=30.0):
    """
    Call read_video_from_roi.py as a subprocess to extract ROI color channels from video.
    
    Args:
        video_path (str): Path to input video file
        output_path (str): Path to output data file
        remove_first_n_seconds (int): Number of seconds to remove from the beginning of the output file
        remove_last_n_seconds (int): Number of seconds to remove from the end of the output file
        fps (float): Video frame rate used for time-to-frame trimming conversion
        
    Returns:
        bool: True if successful, False otherwise
    """
    script_dir = os.path.dirname(os.path.abspath(__file__))
    script_path = os.path.join(script_dir, 'Optikk-lab-filer-26', 'read_video_from_roi.py')
    
    try:
        result = subprocess.run(
            ['python3', script_path, video_path, output_path],
            capture_output=True,
            text=True,
            check=True
        )
        print(result.stdout)

        if remove_first_n_seconds > 0 or remove_last_n_seconds > 0:
            _trim_output_file(output_path, fps, remove_first_n_seconds, remove_last_n_seconds)

        return True
    except subprocess.CalledProcessError as e:
        print(f"Error running read_video_from_roi.py: {e.stderr}")
        return False
    except FileNotFoundError:
        print(f"Script not found at {script_path}")
        return False

In [ ]:
#example usage
from extractColorChannels import run_video_roi_extraction
from plotValues import plot_values
from fft import plot_fft
from snr import snr_from_file_all

#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/fingerEven77bpmTest2.mp4"

video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/panne/panneEven70bpm.mp4"

#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/pulsVari/fingerEven73bpmLav.mp4"
#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/pulsVari/fingerEven100bpmHoy.mp4"

#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/temp/fingerEven66bpmVarm.mp4"
#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/temp/fingerEven74bpmKald.mp4"

#video_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/videofiler/heimeTesting/robusthetstester/avstand/fingerEven68bpm10cm.mp4"

output_file = "/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/outputFiler/colorChannels.txt"
defineNewROI = True

fps = 30

removeFirstNSeconds = 0 # Remove first N seconds of signal
removeLastNSeconds = 0  # Remove last N seconds of signal

if defineNewROI:
    run_video_roi_extraction(video_file, output_file, removeFirstNSeconds, removeLastNSeconds)

plot_values(output_file)

peak_red, peak_green, peak_blue = plot_fft(output_file, fps, .5, 6, True)

print("-----------------------------------")
print("Puls i Hz og bpm for hver kanal:")
print(f'\t Rød kanal: {peak_red:.2f} Hz, {peak_red * 60:.2f} bpm')
print(f'\t Grønn kanal: {peak_green:.2f} Hz, {peak_green * 60:.2f} bpm')
print(f'\t Blå kanal: {peak_blue:.2f} Hz, {peak_blue * 60:.2f} bpm')


print("-----------------------------------")
result = snr_from_file_all(output_file, skip_header=0)
print("SNR for hver kanal:")
print(f'\t Rød kanal: {result[0]:.2f} dB')
print(f'\t Grønn kanal: {result[1]:.2f} dB')
print(f'\t Blå kanal: {result[2]:.2f} dB')

In [2]:
#fft
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import os

#FFT analysis of each color channel, to find the dominant frequencies in the signal
def plot_fft(data_path, fps = 30, lowerbound = 0, upperbound = 6, normalizeChannelsSeparately=True):
    data = np.genfromtxt(data_path, delimiter=" ", skip_header=1)
    #print(data)  # Print the loaded data to verify its structure
    if data.ndim != 2 or data.shape[1] < 3:
        print(f"Error: Data shape is incorrect. Expected (n, 3), got {data.shape}")
        exit()

    red_channel = data[:, 0]
    green_channel = data[:, 1]
    blue_channel = data[:, 2]

    #Filter
    def bandpass_filter(signal, fs, lowcut=0.5, highcut=3, order=4):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = butter(order, [low, high], btype='band')
        return filtfilt(b, a, signal)
    
    red_channel = bandpass_filter(red_channel, fps)
    green_channel = bandpass_filter(green_channel, fps)
    blue_channel = bandpass_filter(blue_channel, fps)

    #Number of samples
    N = len(red_channel)
    #Sampling frequency    fs = fps
    #Sampling period     T = 1.0 / fps
    fs = fps
    T = 1.0 / fs

    #FFT, with zero padding to next power of 2 for better frequency resolution, and a hamming window to reduce spectral leakage. The FFT is computed for each color channel separately, and the magnitude spectrum is plotted. The dominant frequency in the area of interest (0.5-4 Hz) is identified for each channel and printed out.
    N_padded = 2 ** int(np.ceil(np.log2(N)))
    # Apply Hamming window to reduce spectral leakage
    window = np.hamming(N)
    red_channel = red_channel * window
    green_channel = green_channel * window
    blue_channel = blue_channel * window

    freqs = np.fft.rfftfreq(N_padded, d=T)
    red_fft = np.fft.rfft(red_channel, n=N_padded)
    green_fft = np.fft.rfft(green_channel, n=N_padded)
    blue_fft = np.fft.rfft(blue_channel, n=N_padded)

    red_magnitude = np.abs(red_fft) / N_padded
    green_magnitude = np.abs(green_fft) / N_padded
    blue_magnitude = np.abs(blue_fft) / N_padded

    #Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(freqs, red_magnitude, label='Red Channel', color='red', alpha=0.5)
    plt.plot(freqs, green_magnitude, label='Green Channel', color='green')
    plt.plot(freqs, blue_magnitude, label='Blue Channel', color='blue', alpha=0.5)
    plt.title('Magnitude Spectrum of Color Channels')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.xlim(lowerbound, upperbound) #Focus on frequencies up to 6 Hz, which is relevant for heart rate analysis
    plt.legend()
    plt.grid()
    plt.show()

    #Find the peak frequency in each channel
    peak_freq_red = freqs[np.argmax(red_magnitude)]
    peak_freq_green = freqs[np.argmax(green_magnitude)]
    peak_freq_blue = freqs[np.argmax(blue_magnitude)]

    return peak_freq_red, peak_freq_green, peak_freq_blue


def oldFFT(data_path, fps = 40, lowerbound = 0, upperbound = 6, normalizeChannelsSeparately=True):
    data = np.genfromtxt(data_path, delimiter=" ", skip_header=1)
    print(data)  # Print the loaded data to verify its structure
    if data.ndim != 2 or data.shape[1] < 3:
        print(f"Error: Data shape is incorrect. Expected (n, 3), got {data.shape}")
        exit()

    time = np.arange(len(data)) / fps  # Assuming each row corresponds to a frame at fps frames per second
    red_channel = data[:, 0]
    green_channel = data[:, 1]
    blue_channel = data[:, 2]

    cut = 3*fps  #Cut the first 3 seconds of data, to remove artifacts from starting the recording
    red_channel = red_channel[cut:]
    green_channel = green_channel[cut:]
    blue_channel = blue_channel[cut:]

    #Remove DC offset (mean value) from each channel separately.
    red_channel = red_channel - np.mean(red_channel)
    green_channel = green_channel - np.mean(green_channel)
    blue_channel = blue_channel - np.mean(blue_channel)

    #Filtering the signal 
    lowCut = 0.5  #Hz
    highCut = 3   #Hz
    b, a = butter(3, [lowCut/(fps/2), highCut/(fps/2)], btype='band')
    red_channel = filtfilt(b, a, red_channel)
    green_channel = filtfilt(b, a, green_channel)
    blue_channel = filtfilt(b, a, blue_channel)

    # if normalizeChannelsSeparately:
    #     red_channel = red_channel / np.max(np.abs(red_channel))
    #     green_channel = green_channel / np.max(np.abs(green_channel))
    #     blue_channel = blue_channel / np.max(np.abs(blue_channel))
    
    #FFT analysis, with windowing and zero padding to get better frequency resolution. The FFT is computed for each color channel separately, and the magnitude spectrum is plotted. The dominant frequency in the area of interest (0.5-4 Hz) is identified for each channel and printed out.
    N = len(red_channel)
    zeroPaddingFactor = 16  #Increase this to get better frequency resolution, but it also increases computation time. A value of 4 means that the FFT will be computed on a signal that is 4 times longer than the original, with zero padding.
    nfft = zeroPaddingFactor * N
    red_channel = np.concatenate((red_channel, np.zeros(nfft - N)))
    green_channel = np.concatenate((green_channel, np.zeros(nfft - N)))
    blue_channel = np.concatenate((blue_channel, np.zeros(nfft - N)))

    #window = np.hanning(N)  #Hanning window to reduce spectral leakage
    red_fft = np.fft.rfft(red_channel)
    green_fft = np.fft.rfft(green_channel)
    blue_fft = np.fft.rfft(blue_channel)
    freqs = np.fft.rfftfreq(nfft, d=1/fps)

    areaOfInterest = (freqs >= lowerbound) & (freqs <= upperbound)
    freqs = freqs[areaOfInterest]
    red_magnitude = red_fft[areaOfInterest]
    green_magnitude = green_fft[areaOfInterest]
    blue_magnitude = blue_fft[areaOfInterest]

    peak_freq_red = freqs[np.argmax(red_magnitude)]
    peak_freq_green = freqs[np.argmax(green_magnitude)]
    peak_freq_blue = freqs[np.argmax(blue_magnitude)]

    return peak_freq_red, peak_freq_green, peak_freq_blue

#example usage:
if __name__ == "__main__":
    plot_fft("/Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/outputFiler/colorChannels.txt")

FileNotFoundError: /Users/evenfinnoy/Documents/Skule/VSCODE/ELSYSS6/Sensor/Labber/Labb3/outputFiler/colorChannels.txt not found.

In [4]:
#snr
import numpy as np


def _robust_noise_std(signal, trim_percent=5.0):
	if signal.size == 0:
		raise ValueError("Signal is empty")
	if trim_percent < 0 or trim_percent >= 50:
		raise ValueError("trim_percent must be in [0, 50)")

	sorted_vals = np.sort(signal)
	trim = int(round((trim_percent / 100.0) * sorted_vals.size))
	if trim == 0:
		trimmed = sorted_vals
	else:
		trimmed = sorted_vals[trim:-trim]
	if trimmed.size == 0:
		raise ValueError("Trim removed all samples")
	return float(np.std(trimmed, ddof=0))


def snr_from_file(
	data_path,
	column=0,
	delimiter=" ",
	skip_header=1,
	trim_percent=5.0,
	use_abs_max=True,
):
	"""
	Compute SNR as max amplitude divided by robust noise std.

	Args:
		data_path (str): Path to data file.
		column (int): Column index for signal.
		delimiter (str): File delimiter.
		skip_header (int): Lines to skip at top of file.
		trim_percent (float): Percent to trim from both ends for noise std.
		use_abs_max (bool): Use max absolute amplitude if True.

	Returns:
		dict: {"snr": float, "max_amp": float, "noise_std": float}
	"""
	data = np.genfromtxt(data_path, delimiter=delimiter, skip_header=skip_header)
	if data.ndim == 1:
		signal = data
	else:
		if column < 0 or column >= data.shape[1]:
			raise ValueError("column index out of range")
		signal = data[:, column]

	if signal.size == 0:
		raise ValueError("Signal is empty")

	max_amp = float(np.max(np.abs(signal)) if use_abs_max else np.max(signal))
	noise_std = _robust_noise_std(signal, trim_percent=trim_percent)
	if noise_std == 0:
		raise ValueError("Noise std is zero; SNR is undefined")

	return {
		"snr": max_amp / noise_std,
		"max_amp": max_amp,
		"noise_std": noise_std,
	}


def snr_from_file_all(
	data_path,
	delimiter=" ",
	skip_header=1,
	trim_percent=5.0,
	use_abs_max=True,
):
	"""
	Compute SNR for all columns in a file.

	Returns:
		list: SNR values per column (or one value for 1D data).
	"""
	data = np.genfromtxt(data_path, delimiter=delimiter, skip_header=skip_header)
	if data.ndim == 1:
		signal = data
		max_amp = float(np.max(np.abs(signal)) if use_abs_max else np.max(signal))
		noise_std = _robust_noise_std(signal, trim_percent=trim_percent)
		if noise_std == 0:
			raise ValueError("Noise std is zero; SNR is undefined")
		return [max_amp / noise_std]

	results = []
	for idx in range(data.shape[1]):
		signal = data[:, idx]
		if signal.size == 0:
			raise ValueError("Signal is empty")
		max_amp = float(np.max(np.abs(signal)) if use_abs_max else np.max(signal))
		noise_std = _robust_noise_std(signal, trim_percent=trim_percent)
		if noise_std == 0:
			raise ValueError("Noise std is zero; SNR is undefined")
		results.append(max_amp / noise_std)

	return results